# 01 — Residential Mortgage LGD Model

**Australian APRA-Style IRB LGD Framework**

This notebook demonstrates a complete residential mortgage LGD workflow aligned with
Australian Big 4 / APRA practice:

| Step | Description |
|------|-------------|
| 1 | Load synthetic mortgage default & workout data |
| 2 | Exploratory data analysis — LGD distributions, cure rates |
| 3 | Segmentation — Standard/Non-standard × LTV band × LMI |
| 4 | Two-stage model — P(Cure) logistic + LGD\|Loss beta regression |
| 5 | Exposure-weighted long-run LGD |
| 6 | Downturn overlay (macro-linked) |
| 7 | Margin of conservatism |
| 8 | APRA overlays — LMI, floors (10%/15%), standard/non-standard |
| 9 | Illustrative RWA and capital linkage |
| 10 | Validation — accuracy, calibration, PSI, out-of-time backtest |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, classification_report
import statsmodels.api as sm

from src.data_generation import generate_mortgage_data
from src.lgd_calculation import MortgageLGDEngine, exposure_weighted_average
from src.validation import (
    accuracy_metrics, discriminatory_power, conservatism_test,
    calibration_by_segment, compute_psi, out_of_time_split,
    out_of_time_backtest, generate_validation_report
)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

## 1. Generate & Load Data

In [ ]:
loans, cashflows = generate_mortgage_data(n_loans=500, seed=42)

print(f"Loans: {loans.shape}")
print(f"Cashflows: {cashflows.shape}")
print(f"\nDefault date range: {loans['default_date'].min()} to {loans['default_date'].max()}")
print(f"Cure rate: {(loans['resolution_type'] == 'Cure').mean():.1%}")
loans.head()

## 2. Exploratory Data Analysis

In [ ]:
# Summary statistics
print("=== Realised LGD Summary ===")
display(loans['realised_lgd'].describe())

print("\n=== By Resolution Type ===")
display(loans.groupby('resolution_type')['realised_lgd'].describe())

In [ ]:
# Bimodal LGD distribution — the key insight driving two-stage modelling
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(loans['realised_lgd'], bins=40, edgecolor='white', alpha=0.8)
axes[0].set_title('All Defaults — Realised LGD')
axes[0].set_xlabel('LGD')
axes[0].axvline(loans['realised_lgd'].mean(), color='red', ls='--', label=f"Mean: {loans['realised_lgd'].mean():.2%}")
axes[0].legend()

cures = loans[loans['resolution_type'] == 'Cure']['realised_lgd']
losses = loans[loans['resolution_type'] == 'Property Sale']['realised_lgd']

axes[1].hist(cures, bins=30, edgecolor='white', alpha=0.8, color='green')
axes[1].set_title(f'Cures (n={len(cures)}) — Near-Zero LGD')
axes[1].set_xlabel('LGD')

axes[2].hist(losses, bins=30, edgecolor='white', alpha=0.8, color='salmon')
axes[2].set_title(f'Property Sales (n={len(losses)}) — Loss Cases')
axes[2].set_xlabel('LGD')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_lgd_bimodal.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n→ Bimodal distribution confirms need for TWO-STAGE model (cure vs loss).")

In [ ]:
# LTV at default vs LGD — the primary risk driver
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Colour by resolution
for res, colour in [('Cure', 'green'), ('Property Sale', 'salmon')]:
    mask = loans['resolution_type'] == res
    axes[0].scatter(loans.loc[mask, 'ltv_at_default'], loans.loc[mask, 'realised_lgd'],
                    alpha=0.4, s=15, label=res, color=colour)
axes[0].set_xlabel('LTV at Default')
axes[0].set_ylabel('Realised LGD')
axes[0].set_title('LTV at Default vs Realised LGD')
axes[0].legend()

# Boxplot by LTV band
engine = MortgageLGDEngine()
segmented = engine.segment_loans(loans)
segmented.boxplot(column='realised_lgd', by='ltv_band', ax=axes[1])
axes[1].set_title('Realised LGD by LTV Band')
axes[1].set_xlabel('LTV Band')
axes[1].set_ylabel('Realised LGD')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_ltv_vs_lgd.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Key driver analysis
print("=== Average LGD by Key Segments ===")
for col in ['mortgage_class', 'property_type', 'occupancy', 'loan_type', 'state']:
    print(f"\n--- {col} ---")
    summary = loans.groupby(col).agg(
        count=('realised_lgd', 'size'),
        cure_rate=('resolution_type', lambda x: (x == 'Cure').mean()),
        mean_lgd=('realised_lgd', 'mean'),
        mean_ead=('ead', 'mean'),
    ).round(4)
    display(summary)

## 3. Segmentation

Australian mortgage LGD segmentation hierarchy:
```
Level 1: Standard vs Non-Standard  (APRA requirement)
  Level 2: LTV band at default     (primary risk driver)
    Level 3: LMI eligible           (material recovery impact)
```

In [ ]:
# Exposure-weighted long-run LGD by primary segments
engine = MortgageLGDEngine()
lr_lgd = engine.compute_long_run_lgd(loans, segments=['mortgage_class', 'ltv_band'])

print("=== Exposure-Weighted Long-Run LGD ===")
display(lr_lgd)

# Pivot for heatmap
pivot = lr_lgd.pivot(index='mortgage_class', columns='ltv_band', values='lgd_long_run')
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='YlOrRd', ax=ax)
ax.set_title('Exposure-Weighted Long-Run LGD: Mortgage Class × LTV Band')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_segment_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Two-Stage LGD Model

**Stage 1 — P(Cure)**: Logistic regression predicting whether default resolves as a cure.  
**Stage 2 — E[LGD|Loss]**: OLS on logit-transformed LGD for non-cure cases.  
**Combined**: `E[LGD] = P(Cure) × LGD_cure + (1 - P(Cure)) × E[LGD|Loss]`

In [ ]:
# Prepare modelling data
model_df = loans.copy()
model_df['is_cure'] = (model_df['resolution_type'] == 'Cure').astype(int)

# Encode categoricals
model_df['is_non_standard'] = (model_df['mortgage_class'] == 'Non-Standard').astype(int)
model_df['is_investor'] = (model_df['occupancy'] == 'Investor').astype(int)
model_df['is_io'] = (model_df['loan_type'] == 'Interest-Only').astype(int)
model_df['is_unit'] = (model_df['property_type'] == 'Unit').astype(int)

features = ['ltv_at_default', 'credit_score', 'dti', 'seasoning_months',
            'lmi_flag', 'is_non_standard', 'is_investor', 'is_io', 'is_unit']

X = model_df[features].values
y_cure = model_df['is_cure'].values

In [ ]:
# STAGE 1: Logistic regression for P(Cure)
cure_model = LogisticRegression(max_iter=1000, random_state=42)
cure_proba = cross_val_predict(cure_model, X, y_cure, cv=5, method='predict_proba')[:, 1]

cure_model.fit(X, y_cure)

# AUC
cure_auc = roc_auc_score(y_cure, cure_proba)
print(f"Stage 1 — Cure Model AUC (5-fold CV): {cure_auc:.4f}")

# Coefficients
cure_coefs = pd.DataFrame({
    'Feature': features,
    'Coefficient': cure_model.coef_[0].round(4),
}).sort_values('Coefficient', ascending=False)
print("\nCure Model Coefficients:")
display(cure_coefs)

In [ ]:
# STAGE 2: LGD | Loss — regression on non-cure cases
loss_mask = model_df['is_cure'] == 0
loss_df = model_df[loss_mask].copy()

# Logit transform (bounded 0.001 to 0.999)
loss_df['lgd_clipped'] = loss_df['realised_lgd'].clip(0.001, 0.999)
loss_df['lgd_logit'] = np.log(loss_df['lgd_clipped'] / (1 - loss_df['lgd_clipped']))

X_loss = sm.add_constant(loss_df[features].values)
y_loss = loss_df['lgd_logit'].values

loss_model = sm.OLS(y_loss, X_loss).fit()
print("Stage 2 — LGD|Loss OLS (logit-transformed)")
print(f"R-squared: {loss_model.rsquared:.4f}")
print(f"Adj. R-squared: {loss_model.rsquared_adj:.4f}")

# Summary table
loss_coefs = pd.DataFrame({
    'Feature': ['const'] + features,
    'Coefficient': loss_model.params.round(4),
    'P-value': loss_model.pvalues.round(4),
})
display(loss_coefs)

In [ ]:
# COMBINED: E[LGD] = P(Cure) × LGD_cure + (1 - P(Cure)) × E[LGD|Loss]
LGD_CURE = 0.01  # ~1% for workout costs on cured loans

# Predict LGD|Loss for ALL loans (using logit-inverse)
X_all = sm.add_constant(model_df[features].values)
lgd_logit_pred = loss_model.predict(X_all)
lgd_loss_pred = 1 / (1 + np.exp(-lgd_logit_pred))  # inverse logit

# P(Cure) for all loans (fitted model)
p_cure = cure_model.predict_proba(model_df[features].values)[:, 1]

# Combined prediction
model_df['p_cure'] = p_cure
model_df['lgd_loss_pred'] = lgd_loss_pred
model_df['lgd_predicted'] = p_cure * LGD_CURE + (1 - p_cure) * lgd_loss_pred

print("=== Two-Stage Model Predictions ===")
print(f"Mean P(Cure): {p_cure.mean():.2%}")
print(f"Mean LGD|Loss: {lgd_loss_pred[model_df['is_cure'] == 0].mean():.2%}")
print(f"Mean Combined LGD: {model_df['lgd_predicted'].mean():.2%}")
print(f"Actual Mean LGD:   {model_df['realised_lgd'].mean():.2%}")

In [ ]:
# Predicted vs actual scatter
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(model_df['realised_lgd'], model_df['lgd_predicted'], alpha=0.3, s=15)
ax.plot([0, 1], [0, 1], 'r--', lw=1.5, label='Perfect calibration')
ax.set_xlabel('Actual Realised LGD')
ax.set_ylabel('Predicted LGD (Two-Stage)')
ax.set_title('Two-Stage Model: Predicted vs Actual')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_pred_vs_actual.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5–8. Regulatory Pipeline: Long-Run → Downturn → MoC → APRA Overlays

In [ ]:
# Use the model-predicted LGD as input to regulatory pipeline
# (In practice, the bank would use the two-stage model's output)
loans_for_reg = model_df.copy()

engine = MortgageLGDEngine(downturn_scalar=1.15, moc=0.02)

# Apply full APRA overlay chain
result = engine.apply_apra_overlays(loans_for_reg)

print("=== Regulatory Pipeline Summary ===")
pipeline_cols = ['realised_lgd', 'lgd_downturn', 'lgd_with_moc', 'lgd_after_lmi', 'lgd_final']
display(result[pipeline_cols].describe().round(4))

print("\n=== Exposure-Weighted Portfolio Averages ===")
for col in pipeline_cols:
    ewa = exposure_weighted_average(result, lgd_col=col, ead_col='ead')
    print(f"  {col:25s}: {ewa:.4%}")

In [ ]:
# Waterfall: Realised → Downturn → MoC → LMI → Floor → Final
pipeline_means = {col: exposure_weighted_average(result, col) for col in pipeline_cols}

fig, ax = plt.subplots(figsize=(10, 5))
labels = ['Realised\nLGD', 'Downturn\n(×1.15)', '+ MoC\n(+2pp)', 'LMI\nAdjustment', 'Final\n(Floor Applied)']
values = [pipeline_means[c] for c in pipeline_cols]
colors = ['#3498db', '#e67e22', '#e74c3c', '#2ecc71', '#8e44ad']

bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.2%}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Exposure-Weighted LGD')
ax.set_title('Mortgage LGD Regulatory Pipeline — APRA Overlay Waterfall')
ax.set_ylim(0, max(values) * 1.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_waterfall.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# LGD by mortgage class — Standard vs Non-Standard floor impact
for mc in ['Standard', 'Non-Standard']:
    subset = result[result['mortgage_class'] == mc]
    floor = 0.10 if mc == 'Standard' else 0.15
    floored_pct = (subset['lgd_after_lmi'] < floor).mean()
    print(f"\n{mc} Mortgages (n={len(subset)}):")
    print(f"  Mean Realised LGD:  {subset['realised_lgd'].mean():.2%}")
    print(f"  Mean Final LGD:     {subset['lgd_final'].mean():.2%}")
    print(f"  Floor ({floor:.0%}) binding: {floored_pct:.1%} of loans")

## 9. Illustrative RWA & Capital

In [ ]:
result_rwa = engine.compute_illustrative_rwa(result, pd_estimate=0.02)

print("=== Illustrative Capital Metrics ===")
print(f"Total EAD:              ${result_rwa['ead'].sum():>15,.0f}")
print(f"Total RWA (pre-APRA):   ${result_rwa['rwa'].sum():>15,.0f}")
print(f"Total RWA (post-APRA):  ${result_rwa['rwa_after_apra_scalar'].sum():>15,.0f}")
print(f"APRA Scalar Impact:     +{(result_rwa['rwa_after_apra_scalar'].sum() / result_rwa['rwa'].sum() - 1):.1%}")
print(f"Average Risk Weight:    {(result_rwa['rwa_after_apra_scalar'].sum() / result_rwa['ead'].sum()):.2%}")

## 10. Validation & Backtesting

In [ ]:
# Full validation report
segmented_result = engine.segment_loans(result)
val_report = generate_validation_report(
    segmented_result,
    actual_col='realised_lgd',
    predicted_col='lgd_final',
    segment_col='ltv_band',
    date_col='default_date'
)

print("=== Accuracy ===")
for k, v in val_report['accuracy'].items():
    print(f"  {k}: {v}")

print("\n=== Discriminatory Power ===")
for k, v in val_report['discriminatory_power'].items():
    print(f"  {k}: {v}")

print("\n=== Conservatism Test ===")
for k, v in val_report['conservatism'].items():
    print(f"  {k}: {v}")

In [ ]:
# Calibration by LTV band
print("=== Calibration by LTV Band ===")
display(val_report.get('calibration', 'No calibration available'))

In [ ]:
# PSI stability
if 'stability_psi' in val_report:
    psi_result = val_report['stability_psi']
    print(f"PSI: {psi_result['PSI']:.4f} — {psi_result['Interpretation']}")
    display(psi_result['Detail'])

In [ ]:
# Out-of-time backtest
if 'out_of_time' in val_report:
    oot = val_report['out_of_time']
    print(f"Train: {oot['train_size']} loans | Test: {oot['test_size']} loans")
    print(f"\nOut-of-Time Accuracy:")
    for k, v in oot['accuracy'].items():
        print(f"  {k}: {v}")
    print(f"\nOut-of-Time Conservatism:")
    for k, v in oot['conservatism'].items():
        print(f"  {k}: {v}")

In [ ]:
# Save outputs
result_rwa.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mortgage_lgd_results.csv'), index=False)
lr_lgd.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mortgage_segment_summary.csv'), index=False)

print("Outputs saved to outputs/tables/")